# F30 Batch Statistics

Batch notebook for feature extraction, CSV export, and per-feature distribution plots.


In [11]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde
workspace = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(workspace / 'src'))
from f30_fea import AnalysisParams, analyze_directory
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.unicode_minus'] = False

%matplotlib qt
import matplotlib as mpl

mpl.rcParams['font.family'] = 'Times New Roman'     # 普通文本
mpl.rcParams['mathtext.fontset'] = 'custom'         # 数学文本用自定义字体
mpl.rcParams['mathtext.rm'] = 'Times New Roman'     # 正常(math roman)
mpl.rcParams['mathtext.it'] = 'Times New Roman:italic'
mpl.rcParams['mathtext.bf'] = 'Times New Roman:bold'

plt.rcParams["font.family"] = "Times New Roman, SimSun" # 显示汉字
plt.rcParams["font.size"] = 16 # 字号

In [7]:
data_root = workspace / 'data' / 'f130a-pure'
output_dir = workspace / 'outputs' / 'f30_batch_statistics'
output_dir.mkdir(parents=True, exist_ok=True)
# params = AnalysisParams(highpass_hz=1_000.0, analysis_band_hz=(4_000.0, 40_000.0), envelope_smooth_ms=0.20, stft_window_ms=0.64, stft_overlap=0.875, f0_search_hz=(4_000.0, 12_000.0), h2_search_hz=(8_000.0, 24_000.0), h3_search_hz=(12_000.0, 36_000.0), ridge_jump_penalty_hz=2_000.0, ridge_relative_tolerance=0.08, harmonic_presence_ratio_h2=0.08, harmonic_presence_ratio_h3=0.05, f0_harmonic_weights=(1.0, 1.35, 0.55))

params =AnalysisParams(highpass_hz=2_000.0, analysis_band_hz=(5_000.0, 100_000.0), envelope_smooth_ms=1.60, stft_window_ms=0.64, stft_overlap=0.875, 
                        f0_search_hz=(2_000.0, 40_000.0), h2_search_hz=(4_000.0, 80_000.0), h3_search_hz=(12_000.0, 120_000.0), 
                        ridge_jump_penalty_hz=1_000.0, ridge_relative_tolerance=0.08, harmonic_presence_ratio_h2=0.08, harmonic_presence_ratio_h3=0.05, 
                        f0_harmonic_weights=(1.0, 1.35, 0.55))

feature_groups = {'time_domain': ['signal_duration_ms','event_support_ms','rise_ms','decay_ms','peak_pos_ratio','decay_rise_ratio','envelope_peak','envelope_symmetry','bulge_coverage','h2_duration_ms','h3_duration_ms'], 'f0_trajectory': ['f0_start_khz','f0_peak_khz','f0_peak_time_ms','f0_end_khz','f0_mean_khz','f0_span_khz','f0_peak_count','f0_curve_up_ratio','f0_curve_down_ratio','f0_curve_turn_count','f0_arch_r2','f0_arch_score','f0_arch_vertex_ms'], 'harmonics': ['h2_presence_ratio','h2_energy_ratio_to_f0','h2_rel_error_median','h2_longest_duration_ms','h3_presence_ratio','h3_energy_ratio_to_f0','h3_rel_error_median','h3_longest_duration_ms']}


In [8]:
feature_df = analyze_directory(data_root, params=params)
feature_df = feature_df.sort_values('sample_name').reset_index(drop=True)
feature_csv = output_dir / 'f130a_pure_feature_table.csv'
feature_df.to_csv(feature_csv, index=False, encoding='utf-8-sig')
print(f'Saved feature table to: {feature_csv}')
feature_df.head()


Saved feature table to: E:\codes\ZZ-BK\outputs\f30_batch_statistics\f130a_pure_feature_table.csv


,sample_id,sample_name,sample_path,sample_rate_hz,signal_duration_ms,raw_ptp,filtered_rms,filtered_ptp,envelope_peak,event_support_ms,...,f0_arch_score,f0_arch_vertex_ms,h2_presence_ratio,h2_energy_ratio_to_f0,h2_rel_error_median,h3_presence_ratio,h3_energy_ratio_to_f0,h3_rel_error_median,ridge_band_energy_ratio,stft_spectral_peak_khz
0,F130A-FIP-200K-20260324T100102.274,F130A-FIP-200K-20260324T100102.274.npz,E:\codes\ZZ-BK\data\f130a-pure\F130A-FIP-200K-...,200000.0,15.155,1055.543460,0.039201,0.337379,0.071888,10.575,...,0.561572,6.524138,0.979058,0.510299,0.010989,0.188482,0.044127,0.083916,0.947293,18.7500
1,F130A-FIP-200K-20260324T100102.289,F130A-FIP-200K-20260324T100102.289.npz,E:\codes\ZZ-BK\data\f130a-pure\F130A-FIP-200K-...,200000.0,5.170,120.043646,0.021923,0.116508,0.040007,3.540,...,0.549772,2.641626,1.000000,1.153360,0.007813,0.227273,0.481291,0.142857,0.799207,6.2500
2,F130A-FIP-200K-20260324T100102.293,F130A-FIP-200K-20260324T100102.293.npz,E:\codes\ZZ-BK\data\f130a-pure\F130A-FIP-200K-...,200000.0,16.335,927.125218,0.034205,0.153576,0.055694,10.915,...,0.700514,8.592931,1.000000,0.393238,0.018525,0.233010,0.050590,0.095455,0.954291,10.9375
3,F130A-FIP-200K-20260324T100102.309,F130A-FIP-200K-20260324T100102.309.npz,E:\codes\ZZ-BK\data\f130a-pure\F130A-FIP-200K-...,200000.0,15.135,975.808513,0.035674,0.161762,0.055447,10.890,...,0.901171,7.434286,1.000000,0.365673,0.005263,0.073298,0.024140,0.142857,0.968705,17.1875
4,F130A-FIP-200K-20260324T100102.323,F130A-FIP-200K-20260324T100102.323.npz,E:\codes\ZZ-BK\data\f130a-pure\F130A-FIP-200K-...,200000.0,7.780,249.511121,0.028721,0.130666,0.047806,6.260,...,0.855784,3.667116,1.000000,0.716137,0.000000,0.202020,0.375917,0.126984,0.867663,6.2500


In [9]:
numeric_cols = [col for col in feature_df.columns if pd.api.types.is_numeric_dtype(feature_df[col])]
summary_df = feature_df[numeric_cols].describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]).T
summary_csv = output_dir / 'f130a_pure_feature_summary.csv'
summary_df.to_csv(summary_csv, encoding='utf-8-sig')
print(f'Saved summary table to: {summary_csv}')
summary_df.head()


Saved summary table to: E:\codes\ZZ-BK\outputs\f30_batch_statistics\f130a_pure_feature_summary.csv


,count,mean,std,min,10%,25%,50%,75%,90%,max
sample_rate_hz,252.0,200000.000000,0.000000,200000.000000,200000.000000,200000.000000,200000.000000,200000.000000,200000.000000,200000.000000
signal_duration_ms,252.0,11.189147,4.938700,1.400000,3.932000,8.518750,11.382500,14.080000,17.158500,35.680000
raw_ptp,252.0,535.555489,383.447505,23.792050,93.685529,251.958286,452.977048,777.220821,1002.610021,2197.876451
filtered_rms,252.0,0.032499,0.006594,0.016207,0.022571,0.029166,0.033449,0.036875,0.039254,0.054327
filtered_ptp,252.0,0.165708,0.060493,0.084421,0.128451,0.144112,0.156315,0.169016,0.195807,0.562636


In [13]:
def plot_feature_distribution(series: pd.Series, feature_name: str, output_path: Path, bins: int = 20):
    clean = pd.to_numeric(series, errors='coerce').dropna()
    if clean.empty:
        return
    fig, ax1 = plt.subplots(figsize=(7.5, 4.8))
    ax1.hist(clean, bins=bins, color='tab:blue', alpha=0.35, edgecolor='white')
    mean_value = float(clean.mean())
    ax1.axvline(mean_value, color='tab:red', ls='--', lw=1.5, label=f'Mean = {mean_value:.3f}')
    ax1.set_xlabel(feature_name, fontsize=20)
    ax1.set_ylabel('Count', fontsize=20)
    ax1.grid(alpha=0.2)
    ax1.legend(loc='upper right', fontsize=18)
    ax1.tick_params(labelsize=20)
    ax2 = ax1.twinx()
    if clean.nunique() > 1:
        grid = np.linspace(float(clean.min()), float(clean.max()), 300)
        kde = gaussian_kde(clean.to_numpy())
        ax2.plot(grid, kde(grid), color='tab:orange', lw=1.7)
    ax2.set_ylabel('Density', fontsize=20)
    ax2.tick_params(labelsize=20)
    ax1.set_title(f'Distribution of {feature_name}', fontsize=20)
    fig.savefig(output_path, dpi=160, bbox_inches='tight')
    plt.show()
for group_name, feature_names in feature_groups.items():
    group_dir = output_dir / group_name
    group_dir.mkdir(parents=True, exist_ok=True)
    for feature_name in feature_names:
        if feature_name not in feature_df.columns:
            continue
        plot_feature_distribution(feature_df[feature_name], feature_name, group_dir / f'{feature_name}.png')


# 每个子图的物理量中英文、物理意义
# Distribution of signal_duration_ms: 信号持续时间，单位毫秒
# Distribution of event_support_ms：事件支撑时间，单位毫秒
# Distribution of rise_ms：上升时间，单位毫秒
# Distribution of decay_ms：衰减时间，单位毫秒
# Distribution of peak_pos_ratio：峰值位置占比，单位无量纲
# Distribution of decay_rise_ratio：衰减时间与上升时间的比值，单位无量纲
# Distribution of envelope_peak：包络峰值，单位无量
# Distribution of envelope_symmetry：包络对称性，单位无量纲
# Distribution of bulge_coverage：膨胀覆盖率，单位无量纲
# Distribution of h2_duration_ms：第二谐波持续时间，单位毫秒
# Distribution of h3_duration_ms：第三谐波持续时间，单位毫秒
# Distribution of f0_start_khz：基频起始频率，单位千赫
# Distribution of f0_peak_khz：基频峰值频率，单位千赫
# Distribution of f0_peak_time_ms：基频峰值时间，单位毫秒
# Distribution of f0_end_khz：基频结束频率，单位千赫
# Distribution of f0_mean_khz：基频平均频率，单位千赫
# Distribution of f0_span_khz：基频跨度，单位千赫
# Distribution of f0_peak_count：基频峰值数量，单位无量纲
# Distribution of f0_curve_up_ratio：基频曲线上升占比，单位无量纲
# Distribution of f0_curve_down_ratio：基频曲线下降占比，单位无量纲
# Distribution of f0_curve_turn_count：基频曲线转折数量，单位无量纲
# Distribution of f0_arch_r2：基频拱形拟合R²，单位无量纲
# Distribution of f0_arch_score：基频拱形得分，单位无量纲
# Distribution of f0_arch_vertex_ms：基频拱形顶点时间，单位毫秒
# Distribution of h2_presence_ratio：第二谐波存在占比，单位无量纲
# Distribution of h2_energy_ratio_to_f0：第二谐波能量与基频能量的比值，单位无量纲
# Distribution of h2_rel_error_median：第二谐波相对频率误差中位数，单位无量纲
# Distribution of h2_longest_duration_ms：第二谐波最长持续时间，单位毫秒
# Distribution of h3_presence_ratio：第三谐波存在占比，单位无量纲
# Distribution of h3_energy_ratio_to_f0：第三谐波能量与基频能量的比值，单位无量纲
# Distribution of h3_rel_error_median：第三谐波相对频率误差中位数，单位无量纲
# Distribution of h3_longest_duration_ms：第三谐波最长持续时间，单位毫秒
# Distribution of f0_harmonic_weights：基频谐波权重，单位无量纲
# Distribution of f0_search_hz：基频搜索范围，单位赫兹
# Distribution of h2_search_hz：第二谐波搜索范围，单位赫兹
# Distribution of h3_search_hz：第三谐波搜索范围，单位赫兹




C:\Users\QiGh\AppData\Local\Temp\ipykernel_23480\1486193622.py:5: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, ax1 = plt.subplots(figsize=(7.5, 4.8))
